Recovered from the kernel's run history (session of 2026-09-30 16:29): the cells exactly as executed, in execution order.

In [13]:
import regex as re

In [14]:
def get_stats(ids)-> dict: # this is just 
    counts = {}
    for i in range(len(ids)):
        for pair in zip(ids[i], ids[i][1:]):
            counts[pair] = counts.get(pair, 0) + 1
    return counts

In [15]:
def merge(ids, pair, idx):

    out = [[] for _ in range (len(ids))]
    for n in range(len(ids)):
        ids_n = ids[n]
        i = 0
        while i < len(ids_n):
            if i < len(ids_n) - 1 and ids_n[i] == pair[0] and ids_n[i + 1] == pair[1]:
                out[n].append(idx)
                i += 2
            else:
                out[n].append(ids_n[i])
                i += 1
    return out

In [16]:
pattern = re.compile("|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""",
    r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""",
    r"""\s*[\r\n]+""",
    r"""\s+(?!\S)""",
    r"""\s+""",
]))

In [17]:
def train(text, vocab_size, verbose=False):

    #merges: list[tuple[bytes, bytes]]
    text = pattern.findall(text) # this will return a list of string 
    ids = list(chunk.encode("utf-8") for chunk in text)
    vocab = {i: bytes([i]) for i in range(256)}
    print(ids)
    # 257 : b "cd" 
    merges_list = []
    merges = {} # (97,98) -> 257 

    for idx in range(256, vocab_size):
        stats = get_stats(ids)
        if not stats:                                # nothing left to merge
            break
        pair = max(stats, key=lambda p: (stats[p], vocab[p[0]], vocab[p[1]]), )        # tie breaking base on the bytes if want to go ofr the lower use - ? 
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        merges_list.append((vocab[pair[0]], vocab[pair[1]]))
        if verbose:
            print(f"merge {idx - 255}/{vocab_size - 256}: {pair} -> {idx} {vocab[idx]} ({stats[pair]}x)")

    return vocab, merges_list

In [18]:
def encode(text):
    pass

def decode(ids):
    pass

In [19]:
text = "cd cd abab"
vocab, merges = train(text, 269)
print(pattern.findall("cd cd abab"))
print(len(merges))

[b'cd', b' cd', b' abab']
['cd', ' cd', ' abab']
5


In [20]:
with open("../data/small.txt", "r", encoding="utf-8") as f:

    text = f.read()

vocab, merges = train(text, 400)
print(vocab)

[b'u', b" don't", b' have', b' to', b' be', b' scared', b' of', b' the', b' loud', b' dog', b',', b" I'll", b' protect', b' you', b'".', b' The', b' mole', b' felt', b' so', b' safe', b' with', b' the', b' little', b' girl', b'.', b' She', b' was', b' very', b' kind', b' and', b' the', b' mole', b' soon', b' came', b' to', b' trust', b' her', b'.', b' He', b' leaned', b' against', b' her', b' and', b' she', b' kept', b' him', b' safe', b'.', b' The', b' mole', b' had', b' found', b' his', b' best', b' friend', b'.\n', b'<|', b'endoftext', b'|>\n', b'Once', b' upon', b' a', b' time', b',', b' in', b' a', b' warm', b' and', b' sunny', b' place', b',', b' there', b' was', b' a', b' big', b' pit', b'.', b' A', b' little', b' boy', b' named', b' Tom', b' liked', b' to', b' play', b' near', b' the', b' pit', b'.', b' One', b' day', b',', b' Tom', b' lost', b' his', b' red', b' ball', b'.', b' He', b' was', b' very', b' sad', b'.\n', b'Tom', b' asked', b' his', b' friend', b',', b' Sam', b','

In [21]:
with open("../data/small.txt", "r", encoding="utf-8") as f:

    text = f.read()

vocab, merges = train(text, 400)
print(vocab)
print(merges)

[b'u', b" don't", b' have', b' to', b' be', b' scared', b' of', b' the', b' loud', b' dog', b',', b" I'll", b' protect', b' you', b'".', b' The', b' mole', b' felt', b' so', b' safe', b' with', b' the', b' little', b' girl', b'.', b' She', b' was', b' very', b' kind', b' and', b' the', b' mole', b' soon', b' came', b' to', b' trust', b' her', b'.', b' He', b' leaned', b' against', b' her', b' and', b' she', b' kept', b' him', b' safe', b'.', b' The', b' mole', b' had', b' found', b' his', b' best', b' friend', b'.\n', b'<|', b'endoftext', b'|>\n', b'Once', b' upon', b' a', b' time', b',', b' in', b' a', b' warm', b' and', b' sunny', b' place', b',', b' there', b' was', b' a', b' big', b' pit', b'.', b' A', b' little', b' boy', b' named', b' Tom', b' liked', b' to', b' play', b' near', b' the', b' pit', b'.', b' One', b' day', b',', b' Tom', b' lost', b' his', b' red', b' ball', b'.', b' He', b' was', b' very', b' sad', b'.\n', b'Tom', b' asked', b' his', b' friend', b',', b' Sam', b','

In [22]:
with open("../data/small.txt", "r", encoding="utf-8") as f:

    text = f.read()

vocab, merges = train(text, 400)
print(merges)

[b'u', b" don't", b' have', b' to', b' be', b' scared', b' of', b' the', b' loud', b' dog', b',', b" I'll", b' protect', b' you', b'".', b' The', b' mole', b' felt', b' so', b' safe', b' with', b' the', b' little', b' girl', b'.', b' She', b' was', b' very', b' kind', b' and', b' the', b' mole', b' soon', b' came', b' to', b' trust', b' her', b'.', b' He', b' leaned', b' against', b' her', b' and', b' she', b' kept', b' him', b' safe', b'.', b' The', b' mole', b' had', b' found', b' his', b' best', b' friend', b'.\n', b'<|', b'endoftext', b'|>\n', b'Once', b' upon', b' a', b' time', b',', b' in', b' a', b' warm', b' and', b' sunny', b' place', b',', b' there', b' was', b' a', b' big', b' pit', b'.', b' A', b' little', b' boy', b' named', b' Tom', b' liked', b' to', b' play', b' near', b' the', b' pit', b'.', b' One', b' day', b',', b' Tom', b' lost', b' his', b' red', b' ball', b'.', b' He', b' was', b' very', b' sad', b'.\n', b'Tom', b' asked', b' his', b' friend', b',', b' Sam', b','

In [28]:
special_tokens = "<|endoftext|>"

with open("../data/small.txt", "r", encoding="utf-8") as f:
    text = f.read()


tf =re.split(re.escape(special_tokens), text) 
print(len(tf))

# vocab, merges = train(text, 400)

# print(merges)

6
